# Clase 7 - Árboles de Decisión

Vamos a jugar "¿Quién es quién?" para entender qué hace un árbol,
después calculamos Gini y entropía a mano, y terminamos entrenando árboles
sobre datasets que ya conocemos para ver cuándo memorizan de más.

## 1. Jugando a "¿Quién es quién?"

Primero hagámoslo a mano. Alguien elige en secreto un
personaje de la tabla de abajo y el resto le hace preguntas de sí o no
(solo sobre las columnas de la tabla). Anoten **cuántas preguntas** hicieron
falta para adivinar.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

personajes = pd.DataFrame(
    [
        # nombre,  ojos_claros, anteojos, gorra, rubio, barba, sonríe, pelo_largo
        ["Ana",          1, 1, 0, 1, 0, 1, 1],
        ["Bruno",        0, 1, 1, 0, 1, 0, 0],
        ["Carla",        1, 0, 1, 0, 0, 1, 1],
        ["Diego",        0, 0, 0, 1, 1, 1, 0],
        ["Elena",        1, 1, 1, 0, 0, 0, 0],
        ["Facundo",      0, 1, 0, 0, 0, 1, 1],
        ["Gabriela",     1, 0, 0, 1, 0, 0, 1],
        ["Hernán",       0, 0, 1, 1, 0, 0, 0],
        ["Inés",         1, 0, 0, 0, 0, 1, 0],
        ["Julián",       0, 1, 0, 1, 1, 0, 0],
        ["Karina",       1, 1, 0, 0, 0, 0, 1],
        ["Lucas",        0, 0, 1, 0, 1, 1, 0],
        ["Martina",      1, 0, 1, 1, 0, 1, 0],
        ["Nicolás",      0, 0, 0, 0, 1, 0, 1],
        ["Olivia",       1, 1, 1, 1, 0, 0, 1],
        ["Pablo",        0, 1, 1, 1, 0, 1, 0],
    ],
    columns=["nombre", "ojos_claros", "anteojos", "gorra", "rubio", "barba", "sonrie", "pelo_largo"],
)
personajes

Ahora le pedimos a un árbol de decisión que aprenda a predecir el nombre a
partir de las características. Cada personaje es una clase distinta, y el
árbol va a buscar las preguntas que mejor parten el grupo.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree

X = personajes.drop(columns="nombre")
y = personajes["nombre"]

arbol = DecisionTreeClassifier(random_state=0)
arbol.fit(X, y)

print("Preguntas necesarias, como máximo:", arbol.get_depth())
print(export_text(arbol, feature_names=list(X.columns)))

Cómo leer el texto: `anteojos <= 0.50` significa "anteojos = 0", o sea "no
tiene anteojos". Como todas las features son 0/1, cada umbral `<= 0.50` es
exactamente una pregunta de sí o no.

Con 16 personajes, la mejor estrategia posible es partir el grupo a la mitad
en cada pregunta: 16 → 8 → 4 → 2 → 1, o sea **4 preguntas**
($\log_2 16 = 4$). ¿Cuántas usó el árbol? ¿Y ustedes?

### Gini vs. entropía

El árbol necesitó 5 preguntas en el peor caso, una más que el óptimo. ¿Por
qué? Cuando cada personaje es una clase distinta, **cualquier** pregunta
deja el mismo Gini ponderado, así que Gini no tiene forma de
preferir las preguntas que parten el grupo a la mitad.

La entropía, en cambio, mide información en **bits**: adivinar 1 personaje
entre 16 requiere 4 bits, y una pregunta que parte el grupo justo a la mitad
aporta exactamente 1 bit. Probemos:

In [ ]:
arbol = DecisionTreeClassifier(criterion="entropy", random_state=0)
arbol.fit(X, y)

print("Preguntas necesarias, como máximo:", arbol.get_depth())

In [ ]:
print(export_text(arbol, feature_names=list(X.columns)))

Fíjense que ahora la primera pregunta parte a los 16 personajes en dos grupos
de 8, y cada pregunta siguiente vuelve a partir a la mitad.

### El árbol contra ustedes

Piensen un personaje de la tabla y corran la celda de abajo: el árbol les va
a hacer preguntas (respondan `s` o `n`).

In [ ]:
preguntas = {
    "ojos_claros": "¿Tiene ojos claros?",
    "anteojos": "¿Tiene anteojos?",
    "gorra": "¿Tiene gorra?",
    "rubio": "¿Es rubio/a?",
    "barba": "¿Tiene barba?",
    "sonrie": "¿Está sonriendo?",
    "pelo_largo": "¿Tiene el pelo largo?",
}

def jugar(arbol, columnas):
    t = arbol.tree_
    nodo = 0                                    # arrancamos en la raíz
    while t.children_left[nodo] != -1:          # -1 significa "es una hoja"
        feature = columnas[t.feature[nodo]]
        respuesta = input(preguntas[feature] + " (s/n) ")
        if respuesta.strip().lower().startswith("s"):
            nodo = t.children_right[nodo]       # feature = 1 -> rama derecha
        else:
            nodo = t.children_left[nodo]        # feature = 0 -> rama izquierda
    print("¡Tu personaje es", arbol.classes_[t.value[nodo].argmax()] + "!")

jugar(arbol, list(X.columns))

## 2. Midiendo la impureza: Gini y entropía

Para elegir cada pregunta, el árbol necesita un número que diga qué tan
"mezclado" quedó un grupo. Podemos programar las dos medidas que vimos en la teoría. Reciben la cantidad de ejemplos de cada clase en un nodo.

In [ ]:
def gini(conteos):
    p = np.array(conteos) / np.sum(conteos)
    return 1 - np.sum(p ** 2)

def entropia(conteos):
    p = np.array(conteos) / np.sum(conteos)
    p = p[p > 0]                        # log(0) no existe, esos términos valen 0
    return np.sum(p * np.log2(1 / p))   # es lo mismo que -sum(p * log2(p))

for conteos in [[10, 0], [5, 5], [8, 2], [6, 4]]:
    print(f"{conteos}: Gini = {gini(conteos):.3f}, entropía = {entropia(conteos):.3f}")

In [ ]:
p = np.linspace(0, 1, 101)
ginis = [gini([pi, 1 - pi]) for pi in p]
entropias = [entropia([pi, 1 - pi]) for pi in p]

plt.plot(p, ginis, label="Gini")
plt.plot(p, entropias, label="Entropía")
plt.xlabel("Proporción de la clase positiva en el nodo")
plt.ylabel("Impureza")
plt.title("Las dos medidas valen 0 en nodos puros y son máximas en 50/50")
plt.legend()
plt.show()

### ¿Fulbo?

Los mismos 10 días del ejemplo de las slides. ¿Qué conviene preguntar
primero, si llueve o si hace frío?

In [ ]:
dias = pd.DataFrame({
    "llueve":  [1, 1, 1, 1, 0, 0, 0, 0, 0, 0],
    "frio":    [1, 1, 0, 0, 1, 1, 1, 0, 0, 0],
    "jugamos": [1, 0, 0, 0, 1, 1, 0, 1, 1, 1],
})
dias

In [ ]:
def gini_de_pregunta(df, columna, objetivo="jugamos"):
    # Gini promedio de los dos grupos que deja la pregunta, ponderado por tamaño
    total = 0
    for valor, grupo in df.groupby(columna):
        conteos = grupo[objetivo].value_counts().values
        peso = len(grupo) / len(df)
        print(f"  {columna} = {valor}: {len(grupo)} días, Gini = {gini(conteos):.3f}")
        total += peso * gini(conteos)
    return total

print("Gini de la raíz:", round(gini(dias["jugamos"].value_counts().values), 3))
for columna in ["llueve", "frio"]:
    print(f"Pregunta '{columna}':")
    print(f"  -> Gini ponderado = {gini_de_pregunta(dias, columna):.3f}")

Ahora le pedimos a scikit-learn un árbol de **una sola pregunta**
(`max_depth=1`) y verificamos que elige lo mismo que calculamos a mano.

In [ ]:
arbol_futbol = DecisionTreeClassifier(max_depth=1)
arbol_futbol.fit(dias[["llueve", "frio"]], dias["jugamos"])

print(export_text(arbol_futbol, feature_names=["llueve", "frio"], show_weights=True))

## 3. Un árbol sobre un dataset que ya conocemos: iris

En las clases anteriores vimos que las especies de iris se distinguen bastante
bien mirando los pétalos. Veamos qué reglas descubre un árbol por su cuenta.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

iris = load_iris(as_frame=True)
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0, stratify=y
)

arbol_iris = DecisionTreeClassifier(max_depth=3, random_state=0)
arbol_iris.fit(X_train, y_train)

print("Accuracy en train:", round(arbol_iris.score(X_train, y_train), 3))
print("Accuracy en test: ", round(arbol_iris.score(X_test, y_test), 3))

In [ ]:
plt.figure(figsize=(14, 7))
plot_tree(arbol_iris, feature_names=list(X.columns), class_names=list(iris.target_names),
          filled=True, rounded=True, fontsize=9)
plt.show()

Cada caja muestra:

- la **pregunta** (salvo en las hojas),
- el **Gini** del nodo,
- `samples`: cuántos ejemplos de train llegaron hasta ahí,
- `value`: cuántos hay de cada clase,
- `class`: la clase que predeciría el árbol si se frenara en ese nodo.

El color indica la clase mayoritaria, y es más intenso cuanto más puro es el
nodo. Fíjense que la primera pregunta separa a *setosa* perfectamente: es lo
mismo que veíamos en los scatter plots de las clases pasadas.

## 4. Overfitting: el árbol que memoriza

Usamos un dataset de juguete con dos "medialunas" que se meten una dentro de
la otra, con bastante ruido. Así podemos **ver** la frontera de decisión.

In [ ]:
from sklearn.datasets import make_moons
from sklearn.inspection import DecisionBoundaryDisplay

X, y = make_moons(n_samples=400, noise=0.35, random_state=0)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, profundidad in zip(axes, [1, 4, None]):
    arbol = DecisionTreeClassifier(max_depth=profundidad, random_state=0)
    arbol.fit(X_train, y_train)
    DecisionBoundaryDisplay.from_estimator(arbol, X, ax=ax, alpha=0.3, cmap="coolwarm")
    ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap="coolwarm", edgecolor="k", s=20)
    ax.set_title(
        f"max_depth={profundidad}\n"
        f"train: {arbol.score(X_train, y_train):.2f} | test: {arbol.score(X_test, y_test):.2f}"
    )
plt.show()

- Con `max_depth=1` hay una sola pregunta: un único corte recto, demasiado
  simple (**underfitting**).
- Con `max_depth=None` el árbol arma islitas para encerrar cada punto
  ruidoso: 100% en train, pero peor en test (**overfitting**).
- Fíjense en la forma de escalones: todos los cortes son horizontales o
  verticales.

Busquemos la profundidad justa probando muchas:

In [ ]:
profundidades = range(1, 16)
acc_train, acc_test = [], []

for d in profundidades:
    arbol = DecisionTreeClassifier(max_depth=d, random_state=0)
    arbol.fit(X_train, y_train)
    acc_train.append(arbol.score(X_train, y_train))
    acc_test.append(arbol.score(X_test, y_test))

plt.plot(profundidades, acc_train, marker="o", label="train")
plt.plot(profundidades, acc_test, marker="o", label="test")
plt.xlabel("max_depth")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

La curva de train sube siempre (cuanto más profundo, más memoriza). La de
test sube, llega a un máximo y después baja o se estanca: a partir de ahí,
agregar preguntas solo sirve para aprender el ruido.

> Ojo: estamos eligiendo `max_depth` mirando el test set, lo cual es hacer un
> poquito de trampa.

## 5. ¿Qué features usó el árbol? Cáncer de mama otra vez

Una ventaja de los árboles es que
nos dicen cuánto aportó cada feature a reducir la impureza
(`feature_importances_`). Y no hace falta escalar.

In [ ]:
from sklearn.datasets import load_breast_cancer

cancer = load_breast_cancer(as_frame=True)
X = cancer.data
y = cancer.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y
)

arbol_cancer = DecisionTreeClassifier(max_depth=4, random_state=0)
arbol_cancer.fit(X_train, y_train)

print("Accuracy en test:", round(arbol_cancer.score(X_test, y_test), 3))

In [ ]:
importancias = pd.Series(arbol_cancer.feature_importances_, index=X.columns)
importancias = importancias[importancias > 0].sort_values()

importancias.plot.barh()
plt.xlabel("Importancia")
plt.title("Features que usó el árbol")
plt.show()

De las 30 features, el árbol usó apenas unas pocas. La clase pasada, la
regresión logística (con datos escalados) sacaba alrededor de 0.96 de
accuracy en este mismo split. ¿El árbol le gana o pierde? A cambio, el
árbol se puede leer como una lista de reglas que un médico podría revisar.

## Ejercicios

**Ejercicio 1.** Agregar a `personajes` un personaje nuevo que sea idéntico a
"Ana" salvo en una sola característica, volver a entrenar el árbol y mirar
`get_depth()`. ¿Cuántas preguntas hacen falta ahora? ¿Qué pasa si agregan un
personaje exactamente igual a otro?

**Ejercicio 2.** Entrenar el árbol de iris con `criterion="entropy"` en vez
del Gini por defecto y graficarlo. ¿Cambian las preguntas? ¿Cambia el
accuracy en test?

**Ejercicio 3.** En el dataset de las medialunas, en lugar de limitar
`max_depth`, probar con `min_samples_leaf` en `[1, 5, 10, 20, 50]` y graficar
la frontera de cada uno. ¿Qué valor parece el mejor equilibrio?

**Ejercicio 4.** Los árboles son inestables. Entrenar el árbol de cáncer
de mama 5 veces, cambiando el `random_state` del `train_test_split` (0, 1,
2, 3, 4), y anotar cuál es la primera pregunta de cada árbol
(`export_text(...)` o `plot_tree(...)`). ¿Siempre es la misma?

**Ejercicio 5.** Importar `RandomForestClassifier` de
`sklearn.ensemble`, que entrena muchos árboles y los hace votar. Entrenarlo
sobre las medialunas y sobre cáncer de mama, y comparar el accuracy en test
contra un solo árbol. Graficar también su frontera de decisión. ¿Sigue
teniendo forma de escalones?

In [ ]:
# Ejercicio 1: 

In [ ]:
# Ejercicio 2: 

In [ ]:
# Ejercicio 3: 

In [ ]:
# Ejercicio 4: 

In [ ]:
# Ejercicio 5: 